# Assignment 1A — Part A
Run the PDF, cleaning, split, token packing, CPT, perplexity, and forgetting stages.

In [ ]:
# 1. Clone the GitHub repository

In [ ]:
from pathlib import Path
import subprocess
import os

REPO_URL = "https://github.com/tusharchouhan/banking-compliance-llm-assignment-1a.git"
PROJECT = Path("/content/banking-compliance-llm-assignment-1a")

if not (PROJECT / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
print("Project:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())

In [ ]:
!ls

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
#Verify GPU:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
#3. Mount Google Drive and copy PDFs
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil

DRIVE_RAW = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_DATA/raw_pdfs")
LOCAL_RAW = PROJECT / "data" / "raw_pdfs"

if not DRIVE_RAW.exists():
    raise FileNotFoundError(f"Missing Drive folder: {DRIVE_RAW}")

shutil.copytree(DRIVE_RAW, LOCAL_RAW, dirs_exist_ok=True)

pdfs = [
    p for p in LOCAL_RAW.rglob("*")
    if p.is_file() and p.suffix.lower() == ".pdf"
]

total_mb = sum(p.stat().st_size for p in pdfs) / (1024 * 1024)

print("PDF count:", len(pdfs))
print("Total size:", round(total_mb, 2), "MB")
print("First PDF:", pdfs[0] if pdfs else "None")

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd() / 'LLM_ASSIGNMENT' if (Path.cwd()/'LLM_ASSIGNMENT').exists() else Path.cwd()
sys.path.insert(0,str(ROOT))

In [ ]:
# Extract Clean and Split the corpus
!python -m src.data.pdf_extractor

In [ ]:
import pandas as pd

pd.read_csv("reports/tables/extraction_report.csv").head()

In [ ]:
# Run Clean
!python -m src.data.clean_corpus

In [ ]:
pd.read_csv("reports/tables/cleaning_report.csv")

In [ ]:
# Run the 90/10 split:
!python -m src.data.train_eval_split

In [ ]:
#Verify both folders:
!find data/train_corpus -type f -iname "*.txt" | wc -l
!find data/eval_corpus -type f -iname "*.txt" | wc -l

In [ ]:
#5. Tokenize and pack
!python -m src.data.tokenize_and_pack --seq-len 1024


In [ ]:
#Verify
!ls -lh data/train_packed.parquet

In [ ]:
pd.read_csv("reports/tables/token_statistics.csv")

In [ ]:
#6. Model inspection and baseline
!python -m src.evaluation.model_inspection
!python -m src.evaluation.baseline_inference

In [ ]:
# Report
pd.read_csv("reports/tables/model_architecture.csv")

In [ ]:
pd.read_csv("results/baseline_outputs/baseline_outputs.csv")

In [ ]:
#7. Run CPT
!python -m src.training.cpt_train --max-steps 500


In [ ]:
#Generating Loss Report:
!python -m src.evaluation.loss_analysis

In [ ]:
from IPython.display import Image, display

display(Image(filename="reports/figures/loss_curve.png"))

In [ ]:
#8. Run perplexity and forgetting evaluation
from pathlib import Path

eval_files = [
    p for p in Path("data/eval_corpus").rglob("*")
    if p.is_file() and p.suffix.lower() == ".txt"
]

print("Evaluation files:", len(eval_files))

In [ ]:
!python -m src.evaluation.perplexity
!python -m src.evaluation.forgetting_check
!python -m src.evaluation.build_report

In [ ]:
# Display Perplexity
pd.read_csv("results/perplexity/ppl_results.csv")

In [ ]:
# Display forgetting results:
from IPython.display import Markdown, display

display(Markdown(
    Path("results/forgetting_check/forgetting_comparison.md").read_text()
))

In [ ]:
#Save Part A outputs to Google Drive
from pathlib import Path
import shutil

SAVE_DIR = Path("/content/drive/MyDrive/LLM_ASSIGNMENT_PART_A_RESULTS")

items = [
    "data/cleaned_txt",
    "data/train_corpus",
    "data/eval_corpus",
    "data/train_packed.parquet",
    "reports",
    "results",
    "models/cpt_model",
]

for item in items:
    source = PROJECT / item
    destination = SAVE_DIR / item

    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

print("Part A saved to:", SAVE_DIR)

In [ ]:
!python -m src.evaluation.model_inspection
!python -m src.evaluation.baseline_inference
!python -m src.training.cpt_train --max-steps 500
!python -m src.evaluation.loss_analysis
!python -m src.evaluation.perplexity
!python -m src.evaluation.forgetting_check